## AutoShop inkl. MCP Server

In diesem Notebook wird das AutoShop-Webshop-Beispiel in einem Kubernetes-Cluster gestartet und um einen MCP Server erweitert. 

Zuerst werden die benötigten Microservices für Katalog, Kunden, Bestellungen und Webshop im Namespace ms-mcp deployt. 

Zusätzlich wird ein MCP Server gestartet, der als Wrapper um die bestehenden REST APIs des AutoShop-Systems implementiert ist. 

Anschliessend wird der MCP Server über Streamable HTTP getestet, indem eine ClientSession aufgebaut, die verfügbaren Tools abgefragt und ein erstes Tool exemplarisch ausgeführt wird.


In [ ]:
%%bash
kubectl create namespace ms-mcp

In [ ]:

echo "https://"$(cat ~/data/server-ip)":30443"

In [ ]:
%%bash
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/order-deployment.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-mcp  

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-mcp webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

Dazu der MCP Server welcher als Wrapper implementiert ist

In [ ]:
%%bash
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-deployment.yaml 
kubectl apply --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-service.yaml

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-mcp autoshop-mcp-server -o=jsonpath='{ .spec.ports[0].nodePort }')/mcp

### Testen

In [ ]:
import subprocess
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


async def main():
    server_url = get_server_url()

    print(f"Verbinde mit MCP Streamable-HTTP-Server unter: {server_url}")

    try:
        async with streamablehttp_client(server_url) as (read_stream, write_stream, _):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("MCP-Session erfolgreich initialisiert")

                response = await session.list_tools()

                if not response.tools:
                    print("Server antwortet, aber es wurden keine Tools registriert.")
                    return

                print(f"Gefundene Tools ({len(response.tools)}):")
                for tool in response.tools:
                    print(f"- {tool.name}")
                    print(f"  inputSchema: {json.dumps(tool.inputSchema, ensure_ascii=False)}")

                test_tool = None
                for tool in response.tools:
                    required = tool.inputSchema.get("required", [])
                    if not required:
                        test_tool = tool
                        break

                if not test_tool:
                    print("Kein Tool ohne Pflichtparameter gefunden.")
                    print("Rufe ein Tool gezielt mit passenden Argumenten auf.")
                    return

                print(f"\nTeste Tool: {test_tool.name}")
                result = await session.call_tool(test_tool.name, arguments={})

                print("\nAntwort:")
                for item in result.content:
                    print(item)

    except Exception as e:
        print(f"Verbindungs- oder MCP-Fehler: {e}")

In [ ]:
await main()

### Aufräumen

In [ ]:
%%bash
kubectl delete --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-deployment.yaml 
kubectl delete --namespace ms-mcp -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-2-0-deployment/mcp-server-service.yaml
kubectl delete ns ms-mcp